# Task 2: Support Ticket Classification

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Download stopwords if not present
nltk.download('stopwords')

# Set plot style
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Data Generation
Creating a synthetic dataset of customer support tickets.

In [ ]:
data = {
    'ticket_text': [
        # Billing Issues
        "I was charged twice for my subscription.",
        "My credit card was declined but I have funds.",
        "Invoice for last month is incorrect.",
        "I want to update my payment method.",
        "Why is my bill higher this month?",
        
        # Technical Support
        "I cannot log into my account.",
        "The app keeps crashing on startup.",
        "Forgot my password and reset link is not working.",
        "Server error 500 when accessing the dashboard.",
        "The installation failed at 90%.",
        
        # Refund Requests
        "I want a refund for my purchase.",
        "Product is not as described, return please.",
        "Cancel my order and refund the amount.",
        "Accidental purchase, please revert transaction.",
        "Money deducted but service not activated.",
        
        # General Inquiry
        "What are your operating hours?",
        "Do you have a physical store location?",
        "Is there a discount for students?",
        "How can I contact customer support agent?",
        "Where can I find the user manual?"
    ] * 5,  # Duplicate to increase dataset size
    'category': [
        'Billing', 'Billing', 'Billing', 'Billing', 'Billing',
        'Technical', 'Technical', 'Technical', 'Technical', 'Technical',
        'Refund', 'Refund', 'Refund', 'Refund', 'Refund',
        'General', 'General', 'General', 'General', 'General'
    ] * 5
}

df = pd.DataFrame(data)
print(f"Dataset Shape: {df.shape}")
df.head()

## 3. Data Preprocessing
Cleaning the text data: Lowercasing, removing punctuation, and simple stopword removal.

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize and remove stopwords
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(filtered_tokens)

df['clean_text'] = df['ticket_text'].apply(preprocess_text)
df[['ticket_text', 'clean_text']].head()

## 4. Feature Extraction (TF-IDF)
Convert text data into numerical features.

In [ ]:
tfidf = TfidfVectorizer(max_features=1000)
X = tfidf.fit_transform(df['clean_text']).toarray()
y = df['category']

print("Feature Matrix Shape:", X.shape)

## 5. Model Training
Using Naive Bayes classifier.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize Model
model = MultinomialNB()
model.fit(X_train, y_train)

print("Model training complete.")

## 6. Evaluation

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

## 7. Visualizations

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred, labels=model.classes_), annot=True, fmt='d', cmap='Blues', xticklabels=model.classes_, yticklabels=model.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 8. Prediction on New Tickets

In [ ]:
new_tickets = [
    "I cannot access my account, password reset required.",
    "Please cancel my subscription and refund my money.",
    "Where is your head office located?",
    "Double charge on my credit card statement."
]

cleaned_new = [preprocess_text(t) for t in new_tickets]
X_new = tfidf.transform(cleaned_new).toarray()
predictions = model.predict(X_new)

for ticket, category in zip(new_tickets, predictions):
    print(f"Ticket: {ticket} \n-> Predicted Category: {category}\n")